# 03 · Machine Learning
**Flight Analytics Platform**

Models trained in this notebook:
1. **Velocity Predictor** — GradientBoostingRegressor
2. **Flight Pattern Clusterer** — KMeans
3. **Anomaly Detector** — IsolationForest

Evaluation metrics are displayed and models saved to `./models/`.

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import psycopg2
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.ensemble       import GradientBoostingRegressor, IsolationForest
from sklearn.cluster        import KMeans
from sklearn.preprocessing  import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics        import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics        import silhouette_score
import joblib

os.makedirs('/home/jovyan/ml/models', exist_ok=True)
MODEL_DIR = '/home/jovyan/ml/models'
RANDOM_STATE = 42

def get_conn():
    return psycopg2.connect(
        dbname  = os.getenv('POSTGRES_DB',       'flightdw'),
        user    = os.getenv('POSTGRES_USER',     'flightuser'),
        password= os.getenv('POSTGRES_PASSWORD', 'flightpass123'),
        host    = os.getenv('POSTGRES_HOST',     'postgres'),
        port    = os.getenv('POSTGRES_PORT',     '5432'),
    )

def sql(q):
    with get_conn() as c: return pd.read_sql(q, c)

print('✅ Setup complete')

## 1. Load Training Data

In [ ]:
df = sql("""
    SELECT icao24, origin_country,
           avg_velocity_kmh, max_velocity_kmh,
           avg_altitude, max_altitude,
           avg_vertical_rate,
           observation_count, on_ground_count,
           baro_altitude
    FROM main.flight_analytics
    WHERE avg_velocity_kmh IS NOT NULL
      AND avg_altitude      IS NOT NULL
    ORDER BY last_updated DESC
    LIMIT 20000
""")

if len(df) < 200:
    print(f'⚠️  Only {len(df)} rows in DB — generating synthetic data for demo')
    rng = np.random.default_rng(RANDOM_STATE)
    n = 5000
    altitude = rng.uniform(0, 12000, n)
    velocity = 150 + altitude * 0.03 + rng.normal(0, 40, n)
    df = pd.DataFrame({
        'icao24':           [f'SIM{i:05d}' for i in range(n)],
        'origin_country':   rng.choice(['Germany','USA','France','UK'], n),
        'avg_velocity_kmh': np.clip(velocity, 0, 1200),
        'max_velocity_kmh': np.clip(velocity * rng.uniform(1.0,1.3,n), 0, 1400),
        'avg_altitude':     altitude,
        'max_altitude':     altitude * rng.uniform(1.0,1.15,n),
        'avg_vertical_rate':rng.normal(0, 3, n),
        'observation_count':rng.integers(1, 20, n),
        'on_ground_count':  rng.integers(0, 3, n),
        'baro_altitude':    altitude,
    })

# Feature engineering
df['on_ground_ratio'] = (df['on_ground_count'] / df['observation_count'].replace(0, np.nan)).fillna(0)
df['altitude_log']    = np.log1p(df['avg_altitude'].clip(lower=0))
df['speed_delta']     = (df['max_velocity_kmh'] - df['avg_velocity_kmh']).abs().fillna(0)

print(f'Training set: {len(df):,} rows × {df.shape[1]} cols')
df.describe().round(1)

## 2. Model 1 — Velocity Predictor (GBR)

In [ ]:
VEL_FEATURES = ['baro_altitude','avg_vertical_rate','on_ground_ratio','altitude_log']
TARGET       = 'avg_velocity_kmh'

valid = df[VEL_FEATURES + [TARGET]].dropna()
X     = valid[VEL_FEATURES].values
y     = valid[TARGET].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

scaler  = StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_te_s  = scaler.transform(X_te)

gbr = GradientBoostingRegressor(
    n_estimators=200, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=RANDOM_STATE,
)
gbr.fit(X_tr_s, y_tr)

y_pred = gbr.predict(X_te_s)
metrics = {
    'MAE  (km/h)': mean_absolute_error(y_te, y_pred),
    'RMSE (km/h)': np.sqrt(mean_squared_error(y_te, y_pred)),
    'R²':          r2_score(y_te, y_pred),
}
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

joblib.dump(gbr,    os.path.join(MODEL_DIR, 'velocity_predictor.pkl'))
joblib.dump(scaler, os.path.join(MODEL_DIR, 'feature_scaler.pkl'))
print('💾 Models saved.')

In [ ]:
# Actual vs Predicted
sample = min(500, len(y_te))
fig = px.scatter(
    x=y_te[:sample], y=y_pred[:sample],
    labels={'x': 'Actual Speed (km/h)', 'y': 'Predicted Speed (km/h)'},
    title=f'Velocity Predictor — Actual vs Predicted  (R²={metrics["R²"]:.3f})',
    template='plotly_dark', opacity=0.6,
)
lo, hi = min(y_te.min(), y_pred.min()), max(y_te.max(), y_pred.max())
fig.add_shape(type='line', x0=lo, y0=lo, x1=hi, y1=hi,
              line=dict(color='#ef4444', width=1, dash='dot'))
fig.show()

# Feature importance
imp = pd.Series(gbr.feature_importances_, index=VEL_FEATURES).sort_values()
fig2 = px.bar(imp, orientation='h', title='Feature Importances', template='plotly_dark',
              labels={'value':'Importance','index':'Feature'})
fig2.show()

## 3. Model 2 — Flight Pattern Clustering (KMeans)

In [ ]:
CLUS_FEATURES = ['avg_velocity_kmh','avg_altitude','avg_vertical_rate','observation_count']

cdf  = df[CLUS_FEATURES].dropna()
Xc_s = StandardScaler().fit_transform(cdf)

# Elbow method
inertias = []
K_range  = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    km.fit(Xc_s)
    inertias.append(km.inertia_)

fig = px.line(x=list(K_range), y=inertias, markers=True,
              title='Elbow Method — KMeans Inertia vs K',
              labels={'x':'k (clusters)','y':'Inertia'},
              template='plotly_dark')
fig.show()

In [ ]:
K = 6
clu_scaler  = StandardScaler()
Xc_scaled   = clu_scaler.fit_transform(cdf)
kmeans      = KMeans(n_clusters=K, n_init=20, random_state=RANDOM_STATE)
cdf         = cdf.copy()
cdf['cluster'] = kmeans.fit_predict(Xc_scaled)

sil = silhouette_score(Xc_scaled, cdf['cluster'], sample_size=min(1000,len(cdf)))
print(f'Silhouette score (k={K}): {sil:.4f}')

joblib.dump(kmeans,     os.path.join(MODEL_DIR, 'flight_clusterer.pkl'))
joblib.dump(clu_scaler, os.path.join(MODEL_DIR, 'cluster_scaler.pkl'))
print('💾 Cluster model saved.')

fig = px.scatter(
    cdf, x='avg_velocity_kmh', y='avg_altitude',
    color='cluster', color_continuous_scale='Viridis',
    title=f'Flight Pattern Clusters (k={K}, silhouette={sil:.3f})',
    labels={'avg_velocity_kmh':'Avg Speed (km/h)','avg_altitude':'Avg Altitude (m)'},
    template='plotly_dark', opacity=0.6,
)
fig.show()

## 4. Model 3 — Anomaly Detection (IsolationForest)

In [ ]:
ANO_FEATURES = ['avg_velocity_kmh','avg_altitude','avg_vertical_rate',
                'max_velocity_kmh','max_altitude']
adf   = df[ANO_FEATURES].dropna()
ano_scaler = StandardScaler()
Xa    = ano_scaler.fit_transform(adf)

iforest = IsolationForest(n_estimators=200, contamination=0.05,
                          random_state=RANDOM_STATE, n_jobs=-1)
iforest.fit(Xa)

raw_scores   = -iforest.score_samples(Xa)
s_min, s_max = raw_scores.min(), raw_scores.max()
norm_scores  = (raw_scores - s_min) / (s_max - s_min)
adf           = adf.copy()
adf['anomaly_score'] = norm_scores
adf['is_anomaly']    = norm_scores >= 0.7

anomaly_rate = adf['is_anomaly'].mean() * 100
print(f'Anomaly rate (score ≥ 0.7): {anomaly_rate:.1f}%')

joblib.dump(iforest,    os.path.join(MODEL_DIR, 'anomaly_detector.pkl'))
joblib.dump(ano_scaler, os.path.join(MODEL_DIR, 'anomaly_scaler.pkl'))
print('💾 Anomaly model saved.')

fig = px.histogram(adf, x='anomaly_score', nbins=50,
                   color='is_anomaly', color_discrete_map={True:'#ef4444',False:'#22c55e'},
                   title='Anomaly Score Distribution',
                   labels={'anomaly_score':'Anomaly Score'},
                   template='plotly_dark')
fig.add_vline(x=0.7, line_dash='dash', line_color='#f97316',
              annotation_text='Threshold (0.7)')
fig.show()

## 5. Summary Table

In [ ]:
summary = pd.DataFrame([
    {'Model': 'GradientBoostingRegressor', 'Task': 'Velocity Prediction',
     'Key Metric': f"R² = {metrics['R²']:.4f}", 'Artifact': 'velocity_predictor.pkl'},
    {'Model': 'KMeans', 'Task': 'Flight Clustering',
     'Key Metric': f'Silhouette = {sil:.4f}', 'Artifact': 'flight_clusterer.pkl'},
    {'Model': 'IsolationForest', 'Task': 'Anomaly Detection',
     'Key Metric': f'Anomaly rate = {anomaly_rate:.1f}%', 'Artifact': 'anomaly_detector.pkl'},
])
display(summary)